# Pretrain image segmentation model (unet) for feature generation

This notebook purpose is to pretrain a UNet model on the clean training split, on a classic binary segmentation task, in order to learn useful features for the upcoming model for path classification. The pretrained UNet will be used as a feature extractor for the path classification model, and will be fine-tuned on the path classification task in a second step.

For reproducibility, we provide this notebook to allow people to pretrain themselves the UNet model, but we also provide the pretrained model weights in the repository, so that it is not mandatory to run this notebook (see [README](../README.md)) to be able to train the path classification model.

In [ ]:
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import matplotlib.pyplot as plt
import numpy as np
from image_segmentation.data.data_viz import plot_batch

In [ ]:
import os
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["FIVES"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir
splits_filepath = dataset_choice.splits_filepath
ndim = dataset_choice.ndim

In [ ]:
train_batch_size = 4
train_patch_size = 1024

## Data visualization
### Show dataset without data augmentation

In [ ]:
from image_segmentation.data.image_datamodule import ImageDatamodule
from image_segmentation.data.image_dataset import ImageDataset

train_transforms = A.Compose([
    ToTensorV2(),
], additional_targets={"fg_mask": "mask"})

val_transforms = A.Compose([
    ToTensorV2(),
], additional_targets={"fg_mask": "mask"})

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
    val_split_ratio=0.2,
    train_transforms=train_transforms,
    val_transforms=val_transforms,
    test_transforms=val_transforms,
    num_workers=0,
    train_batch_size=train_batch_size,
    val_batch_size=1,
    seed=42,
    shuffle_train=False
)

datamodule.setup()
dataloader = datamodule.train_dataloader()

In [ ]:
for i, batch in enumerate(dataloader):
    plot_batch(batch, 
               ndim=ndim,
               plot_3d_mode="mid_slice")
    break

### Show dataset with data augmentation, crop and normalization

In [ ]:
stats = datamodule.dataset.get_dataset_stats(split_name=train_split)

width, height, n_channels = stats['image_width'], stats['image_height'], stats['n_channels']

mean = stats["foreground"]["mean"]
std = stats["foreground"]["std"]

print(f"Image Size: {width} x {height}")
print(f"Channels: {n_channels}")
print(f"Mean (only on foreground): {mean}")
print(f"Std (only on foreground): {std}")

In [ ]:
from image_segmentation.data.augmentations import build_train_transform, build_val_transform

train_transforms = build_train_transform(ndim, mean, std, train_patch_size)
val_transforms = build_val_transform(ndim, mean, std)

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
    val_split_ratio=0.2,
    train_transforms=train_transforms,
    val_transforms=val_transforms,
    test_transforms=val_transforms,
    num_workers=0,
    train_batch_size=train_batch_size,
    val_batch_size=1,
    seed=42,
    shuffle_train=False
)

datamodule.setup()
dataloader = datamodule.train_dataloader()

In [ ]:
for i, batch in enumerate(dataloader):
    plot_batch(batch, ndim)
    break

### Train data augmentation

### Initialize the BinarySegmentator model with the specified hyperparameters, ready for training on the preprocessed dataset.

In [ ]:
from image_segmentation.models import BinarySegmentator

binary_segmentator = BinarySegmentator(
    lr=1e-3,
    input_channels=3,
    ndim=ndim,
    num_layers=5,
    features_start=32,
    bilinear=True,
    norm_op='instance',
    warmup_epochs=1,
    dropout=0.2,
    kernel_size=5
)

### Initialize the callbacks and pytorch-lightning trainer

In [ ]:
from pytorch_lightning import Trainer
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_dir = dataset_choice.get_checkpoint_dir("unet_pretraining")
os.makedirs(checkpoint_dir, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        dirpath=checkpoint_dir,
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="{data_dir}-best-checkpoint-{epoch:02d}-{val_loss:.4f}"
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=100,
        min_delta=0.00,
        verbose=True,
        mode="min"
    )
]

trainer = Trainer(accelerator='auto', devices="auto", max_epochs=5, precision='16-mixed', callbacks=callbacks)
torch.set_float32_matmul_precision("medium")

## Model training

In [ ]:
trainer.fit(binary_segmentator, datamodule=datamodule)

## Testing the model for binary segmentation task

In [ ]:
from utils.device import get_device

device = get_device()

ckpt_path = dataset_choice.get_first_checkpoint_path("unet_pretraining")

model = BinarySegmentator.load_from_checkpoint(ckpt_path, map_location=device)


In [ ]:
trainer.test(model, datamodule=datamodule)

In [ ]:
import matplotlib.pyplot as plt
import gc

with torch.no_grad():
    for i, batch in enumerate(datamodule.test_dataloader()):
        imgs, masks = batch
        logits = model(imgs)
        binary_preds = (torch.sigmoid(logits) > 0.5).float()

        plot_batch(batch, ndim=ndim, pred=binary_preds, plot_3d_mode="mid_slice")
        
        del imgs, masks, logits, binary_preds
        if i >= 2:
            break

torch.cuda.empty_cache()
gc.collect()

Now that we have a trained model for feature extraction, you can continue on the [Segmentation predictions generation notebook (3)](./03_generate_segmentation_preds.ipynb)